# Document to Audio LangGraph

To keep this completely free while maintaining high quality, you can leverage generous free-tier APIs and open-source local models:

- **Orchestration:** LangGraph (Open-source, free)
- **LLM (Scriptwriter & Fact-Checker):** Gemini 1.5 Flash via Google AI Studio — incredibly generous free tier, natively processes massive documents (up to 1M+ tokens), highly capable of self-correction
- **Text-to-Speech (Audio):** Kokoro-82M or Edge-TTS — stellar, lightweight, open-source TTS model that runs beautifully on standard local CPU for free, producing studio-grade AI voices
- **Parsing:** pypdf or python-docx for document ingestion (Open-source, free)

## Workflow Architecture

Before jumping into the plan, here is how the data and control will flow through your LangGraph state machine.

### The Shared Graph State

Every node in LangGraph reads from and writes to a shared, typed state. For your project, the state will track:

- `document_text` — The raw text extracted from the upload
- `script` — The current draft of the podcast script
- `feedback` — The evaluation notes from the fact-checker
- `is_factual` — A boolean flag (True/False) indicating if it passed the check
- `iteration_count` — An integer tracking how many rewrites have occurred (capped at 3)
- `audio_path` — The file path to the final output audio file

## The Step-by-Step Project Plan

### Phase 1: Environment Setup & Document Parsing

**Goal:** Read the uploaded file and get your environment ready.

**Tasks:**
- Install dependencies: `pip install langgraph langchain-google-genai kokoro soundfile pypdf python-docx`
- Set up your free Google AI Studio API key
- Write a helper utility function using pypdf to extract text from an uploaded document and save it to the graph state

### Phase 2: Defining the Graph Nodes (The Agents)

**Goal:** Code the isolated Python functions that act as your workflow's "steps."

**Tasks:**
- **Scriptwriter Node** — Takes `document_text` (and any existing `feedback`), instructs Gemini to write/adjust the script, and updates the `script` state
- **Fact-Checker Node** — Instructs Gemini to compare the new `script` against the original `document_text`. It outputs a critique (`feedback`), flips `is_factual` to True if accurate, and increments `iteration_count`
- **Audio Generator Node** — Takes the finalized `script` and passes it to Kokoro TTS to compile the audio file

### Phase 3: Implementing Loop Control & Conditional Edges

**Goal:** Stitch the nodes together and enforce the "max 3 rewrites" rule.

**Tasks:**
- Create a conditional routing function (`should_continue`) that looks at `is_factual` and `iteration_count`
- Build and compile the StateGraph using the blueprint below

### Phase 4: Testing & Polish

**Goal:** Run the graph and test the boundaries.

**Tasks:**
- Feed the graph a document with a known tricky or ambiguous statement to force the Fact-Checker to catch it and trigger a rewrite loop
- Verify that the loop gracefully breaks and moves to TTS on the 3rd attempt, even if the script isn't 100% perfect, to avoid infinite API loops

## Graph Flow Diagram

```mermaid
graph TD
    Start([START]) --> GenerateScript["📝 generate_script<br/>Write/rewrite script<br/>from document_text + feedback"]

    GenerateScript --> FactCheck["✓ fact_check_script<br/>Compare script vs document<br/>Update feedback & iteration_count"]

    FactCheck --> Decision{is_factual OR<br/>iteration_count >= 3?}

    Decision -->|No| GenerateScript
    Decision -->|Yes| TTS["🔊 text_to_speech<br/>Convert script to audio<br/>Write audio_path"]

    TTS --> End([END])

    style Start fill:#13751f
    style GenerateScript fill:#0a2f6d
    style FactCheck fill:#0a2f6d
    style Decision fill:#bab407
    style TTS fill:#ba07b1
    style End fill:#f23c0e
```

In [ ]:
# Uncomment this to install all necessary dependencies.
# %pip install langgraph langchain-google-genai pypdf python-docx kokoro soundfile

In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Document Parsing
from pypdf import PdfReader          # PDF
from docx import Document            # DOCX (python-docx package)

# TTS + audio output
from kokoro import KPipeline
import soundfile as sf
import numpy as np

# Load GOOGLE_API_KEY from .env into the environment.
# langchain-google-genai picks it up automatically; we never hardcode the key.
load_dotenv()

# Single shared LLM for both the scriptwriter and the fact-checker.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Kokoro is expensive to construct, so build the pipeline once and reuse it
# across every text_to_speech call (and across rewrite loops).
_tts_pipeline = None

def get_tts_pipeline():
    global _tts_pipeline
    if _tts_pipeline is None:
        _tts_pipeline = KPipeline(lang_code="a")  # 'a' = American English
    return _tts_pipeline


# --- Document parsing helper -------------------------------------------------
def parse_document(path: str) -> str:
    """Extract raw text from a .pdf or .docx file."""
    lower = path.lower()
    if lower.endswith(".pdf"):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if lower.endswith(".docx"):
        doc = Document(path)
        return "\n".join(para.text for para in doc.paragraphs)
    raise ValueError(f"Unsupported file type: {path} (expected .pdf or .docx)")


# 1. Define the State
class PodcastState(TypedDict):
    document_text: str
    document_name: str
    script: str
    feedback: str
    is_factual: bool
    iteration_count: int
    audio_path: str


# Structured output schema for the fact-checker, so is_factual is reliable.
class FactCheck(BaseModel):
    is_factual: bool = Field(description="True if the script is faithful to the source document.")
    feedback: str = Field(description="Specific, actionable critique of any inaccuracies; brief confirmation if accurate.")


# 2. Define Node Functions
def generate_script(state: PodcastState):
    # Gemini writes/rewrites the script from the document and any prior feedback.
    print(f"Generating script... (Iteration {state['iteration_count'] + 1})")

    prompt = (
        "You are a podcast scriptwriter. Using ONLY the information in the source "
        "document below, write a single-narrator podcast script.\n\n"
        "Requirements:\n"
        "- Plain spoken-word prose only. No stage directions, no speaker labels, "
        "no markdown, no headings, no bullet points.\n"
        "- Stay strictly faithful to the source; do not invent facts.\n"
        "- Where possible, quote the source document directly.\n\n"
        f"SOURCE DOCUMENT:\n{state['document_text']}\n"
    )

    feedback = state.get("feedback")
    if feedback:
        prompt += (
            "\nA fact-checker reviewed your previous draft and found issues. "
            "Revise the script to address this critique while keeping it faithful "
            f"to the source:\n{feedback}\n"
        )

    return {"script": llm.invoke(prompt).content}


def fact_check_script(state: PodcastState):
    # Gemini compares the script against the source and returns structured output.
    print("Checking for hallucinations...")

    checker = llm.with_structured_output(FactCheck)
    prompt = (
        "You are a strict fact-checker. Compare the PODCAST SCRIPT against the "
        "SOURCE DOCUMENT.\n"
        "- If the script adds claims absent from the source, or contradicts it, set "
        "is_factual=False and give specific, actionable feedback naming what to fix.\n"
        "- If the script is faithful to the source, set is_factual=True with a brief "
        "confirmation.\n\n"
        f"SOURCE DOCUMENT:\n{state['document_text']}\n\n"
        f"PODCAST SCRIPT:\n{state['script']}\n"
    )

    result = checker.invoke(prompt)
    return {
        "feedback": result.feedback,
        "is_factual": result.is_factual,
        "iteration_count": state["iteration_count"] + 1,
    }


def text_to_speech(state: PodcastState):
    # Pass the finalized script to the local Kokoro TTS engine.
    print("Converting final script to audio...")

    pipeline = get_tts_pipeline()
    chunks = [audio for _graphemes, _phonemes, audio in pipeline(state["script"], voice="af_heart")]
    full_audio = np.concatenate(chunks)

    audio_path = f"{state['document_name']}_podcast.wav"
    sf.write(audio_path, full_audio, 24000)  # Kokoro emits 24 kHz audio
    return {"audio_path": audio_path}


# 3. Conditional Router Function
def should_continue(state: PodcastState):
    # Safety boundary: stop after a pass OR after 3 rewrites (no infinite loops).
    if state["is_factual"] or state["iteration_count"] >= 3:
        return "text_to_speech"
    return "generate_script"


# 4. Build the Graph
workflow = StateGraph(PodcastState)

# Add Nodes
workflow.add_node("generate_script", generate_script)
workflow.add_node("fact_check_script", fact_check_script)
workflow.add_node("text_to_speech", text_to_speech)

# Construct Edges
workflow.add_edge(START, "generate_script")
workflow.add_edge("generate_script", "fact_check_script")

# The magic loop step:
workflow.add_conditional_edges(
    "fact_check_script",
    should_continue,
    {
        "generate_script": "generate_script",
        "text_to_speech": "text_to_speech"
    }
)
workflow.add_edge("text_to_speech", END)

# Compile the Graph
app = workflow.compile()

In [ ]:
# --- Run the pipeline --------------------------------------------------------
# Point this at your source document (.pdf or .docx).
DOCUMENT_PATH = r"C:\Users\Zeeshan\Documents\Module_10_Machine_Learning_Study_Guide.docx"
DOCUMENT_NAME = "Module_10_Machine_Learning_Study_Guide"

document_text = parse_document(DOCUMENT_PATH)
print(f"Parsed {len(document_text)} characters from {DOCUMENT_PATH}\n")

# iteration_count must start at 0 — the nodes read it before the first write.
result = app.invoke({"document_text": document_text, "document_name": DOCUMENT_NAME, "iteration_count": 0})

print("\nDone.")
print("Passed fact-check:", result["is_factual"], "| rewrites:", result["iteration_count"])
print("Audio written to:", result["audio_path"])